In [0]:
import gzip, zipfile, io, os
import datetime as dt
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DateType,
                               TimestampType, DoubleType, LongType)

In [0]:
mkt_summary_schema = StructType([
    StructField('trade_date',StringType(),True),
    StructField('symbol',StringType(),True),
    StructField('sector_code',StringType(),True),
    StructField('company_name',StringType(),True),
    StructField('open_raw',StringType(),True),
    StructField('high_raw',StringType(),True),
    StructField('low_raw',StringType(),True),
    StructField('close_raw',StringType(),True),
    StructField("volume_raw",StringType(), True),
    StructField("ldcp_raw",StringType(), True),  
    StructField("extra_1",StringType(), True),   
    StructField("extra_2",StringType(), True),   
    StructField("extra_3",StringType(), True)
])

In [0]:
EXPECTED_PIPES = 12   # 13 fields -> 12 "|" separators                      # NEW

# Schema for quarantine rows (matches ops.quarantine_mkt_summary)            # NEW
quarantine_schema = StructType([
    StructField("raw_record",      StringType(),    True),
    StructField("reason",          StringType(),    True),
    StructField("layer",           StringType(),    True),
    StructField("source_file",     StringType(),    True),
    StructField("file_trade_date", DateType(),      True),
    StructField("batch_id",        StringType(),    True),
    StructField("load_timestamp",  TimestampType(), True),
])

In [0]:
log_schema = StructType([
    StructField("batch_id",         StringType(),    True),
    StructField("layer",            StringType(),    True),
    StructField("load_type",        StringType(),    True),
    StructField("parameter",        StringType(),    True),
    StructField("start_time",       TimestampType(), True),
    StructField("end_time",         TimestampType(), True),
    StructField("duration_seconds", DoubleType(),    True),
    StructField("status",           StringType(),    True),
    StructField("rows_inserted",    LongType(),      True),
    StructField("rows_updated",     LongType(),      True),
    StructField("rows_quarantined", LongType(),      True),
    StructField("error_message",    StringType(),    True),
    StructField("load_timestamp",   TimestampType(), True),
])

In [0]:

def write_log(batch_id, layer, load_type, parameter, start_time, end_time,
              status, rows_inserted=0, rows_updated=0, rows_quarantined=0,
              error_message=None):
    duration = (end_time - start_time).total_seconds()
    row = (batch_id, layer, load_type, parameter, start_time, end_time,
           duration, status, int(rows_inserted), int(rows_updated),
           int(rows_quarantined), error_message, dt.datetime.now())

    (spark.createDataFrame([row], log_schema)
          .write.mode("append")
          .saveAsTable("workspace.ops.pipeline_execution_logs"))